# Library import

In [1]:
import os
import pysam
import pandas as pd
import numpy as np
import pickle
from tqdm.auto import tqdm

# Definition of functions for the application of PRS313

In [2]:
def parse_risk_alleles(riskall_value):
    if pd.isna(riskall_value):
        return []
    if not isinstance(riskall_value, str):
        riskall_value = str(riskall_value)
    return [a.strip() for a in riskall_value.split(",") if a.strip()]


def load_prs_data(prs_path):
    with open(prs_path, "rb") as f:
        return pickle.load(f)


def compute_dosage_all_samples(rec, risk_alleles, samples):
    ref_is_risk = rec.ref in risk_alleles

    risk_alt_indices = []
    if rec.alts is not None:
        for j, alt in enumerate(rec.alts):
            if alt in risk_alleles:
                risk_alt_indices.append(j)

    if not (ref_is_risk or risk_alt_indices):
        return None, "no_risk_allele"

    if ref_is_risk and risk_alt_indices:
        return None, "ambiguous"

    out = {s: None for s in samples}

    if ref_is_risk:
        for s in samples:
            ss = rec.samples[s]
            try:
                ds = ss["DS"]
                sum_alt = float(sum(ds)) if rec.alts else 0.0
                out[s] = 2.0 - sum_alt
            except KeyError:
                gt = ss.get("GT", (None, None))
                if gt == (None, None):
                    out[s] = None
                else:
                    dose = 0
                    for g in gt:
                        if g is None:
                            continue
                        if g == 0:
                            dose += 1
                    out[s] = float(dose)
            except (IndexError, TypeError):
                out[s] = None
        return out, "ok"

    for s in samples:
        ss = rec.samples[s]
        try:
            ds = ss["DS"]
            out[s] = float(sum(ds[j] for j in risk_alt_indices))
        except KeyError:
            gt = ss.get("GT", (None, None))
            if gt == (None, None):
                out[s] = None
            else:
                dose = 0
                for g in gt:
                    if g is None:
                        continue
                    if g > 0 and (g - 1) in risk_alt_indices:
                        dose += 1
                out[s] = float(dose)
        except (IndexError, TypeError):
            out[s] = None

    return out, "ok"


In [3]:
def compute_prs_fast(vcf_file, data_prs, threads=4, show_top_samples=10):
    snps = data_prs["snps"].copy()
    snps_sd = float(data_prs["snps_sd"])
    var1 = float(data_prs["var1"])
    sd1 = float(data_prs["sd1"])

    snps["chromosome"] = snps["chromosome"].astype(str)
    snps["position"] = snps["position"].astype(int)
    snps["risk_set"] = snps["riskall"].apply(lambda x: set(parse_risk_alleles(x)))
    snps["eaf"] = snps["eaf"].astype(float)
    snps["beta"] = snps["beta"].astype(float)

    bcf_in = pysam.VariantFile(vcf_file, threads=threads)
    samples = list(bcf_in.header.samples)

    theta = {s: 0.0 for s in samples}
    non_missing = {s: 0 for s in samples}

    not_found = 0
    ambiguous = 0
    no_risk = 0

    pbar = tqdm(total=len(snps), desc="SNPs", unit="snp")
    for _, row in snps.iterrows():
        chrom = row["chromosome"]
        pos = int(row["position"])
        risk_alleles = row["risk_set"]
        eaf = float(row["eaf"])
        beta = float(row["beta"])

        rec_found = None
        try:
            for rec in bcf_in.fetch(chrom, pos - 1, pos):
                if rec.pos == pos:
                    rec_found = rec
                    break
        except ValueError as e:
            raise ValueError("VCF non indexé ou contig manquant. Indexe avec bcftools index / tabix.") from e

        if rec_found is None:
            not_found += 1
            pbar.update(1)
            pbar.set_postfix(found=f"{len(snps)-not_found}", not_found=not_found, ambiguous=ambiguous, no_risk=no_risk)
            continue

        doses, status = compute_dosage_all_samples(rec_found, risk_alleles, samples)

        if status == "ambiguous":
            ambiguous += 1
            pbar.update(1)
            pbar.set_postfix(found=f"{len(snps)-not_found}", not_found=not_found, ambiguous=ambiguous, no_risk=no_risk)
            continue

        if status == "no_risk_allele":
            no_risk += 1
            pbar.update(1)
            pbar.set_postfix(found=f"{len(snps)-not_found}", not_found=not_found, ambiguous=ambiguous, no_risk=no_risk)
            continue

        twoeaf = 2.0 * eaf
        for s in samples:
            d = doses[s]
            if d is None or (isinstance(d, float) and np.isnan(d)):
                continue
            gi = d - twoeaf
            theta[s] += gi * beta
            non_missing[s] += 1

        pbar.update(1)
        pbar.set_postfix(found=f"{len(snps)-not_found}", not_found=not_found, ambiguous=ambiguous, no_risk=no_risk)

    pbar.close()

    results = {}
    for s in samples:
        prs = float(theta[s])
        deviation = prs / snps_sd
        prs_adj = deviation * sd1
        rr = float(np.exp(prs_adj - var1 / 2.0))
        results[s] = {
            "prs": prs,
            "risk_ratio": rr,
            "num_snps_with_genotypes": int(non_missing[s]),
            "num_snps_total": int(len(snps)),
        }

    top = sorted(non_missing.items(), key=lambda x: x[1], reverse=True)[: int(show_top_samples)]
    top_df = pd.DataFrame(top, columns=["sample", "num_snps_with_genotypes"])

    return results, top_df

# Importation of genetics data

In [3]:
import os

name_of_file_in_bucket = "prs313_all.bcf"

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/prs313_all.bcf...
/ [1 files][ 32.8 MiB/ 32.8 MiB]                                                
Operation completed over 1 objects/32.8 MiB.                                     


[INFO] prs313_all.bcf is successfully downloaded into your working space


In [15]:
%%bash

bcftools index -f prs313_all.bcf
bcftools query -l prs313_all.bcf | grep 2445372


CalledProcessError: Command 'b'\nbcftools index -f prs313_all.bcf\nbcftools query -l prs313_all.bcf | grep 2445372\n'' returned non-zero exit status 1.

# Application of PRS 298

## Run PRS 298

In [7]:
vcf_path = "prs313_all.bcf"
prs_path_298 = "prs_predilife_prs313_aou_298_13-01-2026.pkl"

data_prs_298 = load_prs_data(prs_path_298)

results_298, top_df_298 = compute_prs_fast(vcf_path, data_prs_298, threads=4, show_top_samples=15)

df_results_298_snps = pd.DataFrame(results_298).T
df_results_298_snps

SNPs:   0%|          | 0/298 [00:00<?, ?snp/s]

,prs,risk_ratio,num_snps_with_genotypes,num_snps_total
1000000,1.358896,2.675651,294.0,298.0
1000004,-0.311085,0.694429,295.0,298.0
1000033,-0.650725,0.527822,296.0,298.0
1000039,1.043029,2.073139,294.0,298.0
1000042,-0.115890,0.813016,295.0,298.0
...,...,...,...,...
9999678,0.324153,1.160002,294.0,298.0
9999697,0.457096,1.291495,294.0,298.0
9999715,0.715247,1.590919,294.0,298.0
9999755,-0.344399,0.675992,294.0,298.0


## Formatting results PRS 298

In [8]:
df_results_298_snps.columns = ["prs_298", "risk_ratio_298", "num_snps_with_genotypes_298", "num_snps_total_298"]

In [9]:
df_results_298_snps = df_results_298_snps.reset_index()

In [10]:
df_results_298_snps.head()

,index,prs_298,risk_ratio_298,num_snps_with_genotypes_298,num_snps_total_298
0,1000000,1.358896,2.675651,294.0,298.0
1,1000004,-0.311085,0.694429,295.0,298.0
2,1000033,-0.650725,0.527822,296.0,298.0
3,1000039,1.043029,2.073139,294.0,298.0
4,1000042,-0.115890,0.813016,295.0,298.0
5,1000045,-0.129942,0.803840,294.0,298.0
6,1000059,-0.019636,0.878746,295.0,298.0
7,1000061,0.631381,1.486720,294.0,298.0
8,1000062,0.098761,0.966930,297.0,298.0
9,1000070,0.271501,1.111704,294.0,298.0


# Application of PRS 305

## Run PRS 305

In [12]:
prs_path_305 = "prs_predilife_prs313_aou_305_13-01-2026.pkl"

data_prs_305 = load_prs_data(prs_path_305)

results_305, top_df_305 = compute_prs_fast(vcf_path, data_prs_305, threads=4, show_top_samples=15)

df_results_305_snps = pd.DataFrame(results_305).T
df_results_305_snps

SNPs:   0%|          | 0/305 [00:00<?, ?snp/s]

,prs,risk_ratio,num_snps_with_genotypes,num_snps_total
1000000,1.181928,2.295320,301.0,305.0
1000004,-0.337653,0.681705,302.0,305.0
1000033,-0.684593,0.516677,303.0,305.0
1000039,0.906361,1.841747,301.0,305.0
1000042,-0.254158,0.728729,302.0,305.0
...,...,...,...,...
9999678,0.411185,1.239993,301.0,305.0
9999697,0.467428,1.296982,301.0,305.0
9999715,0.683479,1.541335,301.0,305.0
9999755,-0.317667,0.692677,301.0,305.0


## Formatting results PRS 305

In [13]:
df_results_305_snps.columns = ["prs_305", "risk_ratio_305", "num_snps_with_genotypes_305", "num_snps_total_305"]

In [16]:
df_results_305_snps = df_results_305_snps.reset_index()

In [17]:
df_results_298_snps.head()

,index,prs_298,risk_ratio_298,num_snps_with_genotypes_298,num_snps_total_298
0,1000000,1.358896,2.675651,294.0,298.0
1,1000004,-0.311085,0.694429,295.0,298.0
2,1000033,-0.650725,0.527822,296.0,298.0
3,1000039,1.043029,2.073139,294.0,298.0
4,1000042,-0.115890,0.813016,295.0,298.0


# Application of PRS 311

## Run PRS 311

In [14]:
prs_path_311 = "prs_predilife_prs313_aou_311_13-01-2026.pkl"

data_prs_311 = load_prs_data(prs_path_311)

results_311, top_df_311 = compute_prs_fast(vcf_path, data_prs_311, threads=4, show_top_samples=15)

df_results_311_snps = pd.DataFrame(results_311).T
df_results_311_snps

SNPs:   0%|          | 0/311 [00:00<?, ?snp/s]

,prs,risk_ratio,num_snps_with_genotypes,num_snps_total
1000000,1.287001,2.480273,306.0,311.0
1000004,-0.359895,0.670907,308.0,311.0
1000033,-0.671097,0.524037,307.0,311.0
1000039,0.871333,1.783119,306.0,311.0
1000042,-0.151900,0.791366,308.0,311.0
...,...,...,...,...
9999678,0.398857,1.225389,306.0,311.0
9999697,0.535786,1.366112,307.0,311.0
9999715,0.678675,1.530219,305.0,311.0
9999755,-0.264795,0.723522,306.0,311.0


## Formatting results PRS 311

In [32]:
df_results_311_snps.columns = ["prs_311", "risk_ratio_311", "num_snps_with_genotypes_311", "num_snps_total_311"]

ValueError: Length mismatch: Expected axis has 5 elements, new values have 4 elements

In [18]:
df_results_311_snps = df_results_311_snps.reset_index()

In [33]:
df_results_311_snps.head()

,index,prs_311,risk_ratio_311,num_snps_with_genotypes_311,num_snps_total_311
0,1000000,1.287001,2.480273,306.0,311.0
1,1000004,-0.359895,0.670907,308.0,311.0
2,1000033,-0.671097,0.524037,307.0,311.0
3,1000039,0.871333,1.783119,306.0,311.0
4,1000042,-0.151900,0.791366,308.0,311.0


## Fusion des résultats PRS

In [34]:
df_all_prs = pd.merge(df_results_298_snps, df_results_305_snps, on='index', how='inner')
df_all_prs

,index,prs_298,risk_ratio_298,num_snps_with_genotypes_298,num_snps_total_298,prs_305,risk_ratio_305,num_snps_with_genotypes_305,num_snps_total_305
0,1000000,1.358896,2.675651,294.0,298.0,1.181928,2.295320,301.0,305.0
1,1000004,-0.311085,0.694429,295.0,298.0,-0.337653,0.681705,302.0,305.0
2,1000033,-0.650725,0.527822,296.0,298.0,-0.684593,0.516677,303.0,305.0
3,1000039,1.043029,2.073139,294.0,298.0,0.906361,1.841747,301.0,305.0
4,1000042,-0.115890,0.813016,295.0,298.0,-0.254158,0.728729,302.0,305.0
...,...,...,...,...,...,...,...,...,...
414825,9999678,0.324153,1.160002,294.0,298.0,0.411185,1.239993,301.0,305.0
414826,9999697,0.457096,1.291495,294.0,298.0,0.467428,1.296982,301.0,305.0
414827,9999715,0.715247,1.590919,294.0,298.0,0.683479,1.541335,301.0,305.0
414828,9999755,-0.344399,0.675992,294.0,298.0,-0.317667,0.692677,301.0,305.0


In [35]:
df_all_prs = pd.merge(df_all_prs, df_results_311_snps, on='index', how='inner')
df_all_prs

,index,prs_298,risk_ratio_298,num_snps_with_genotypes_298,num_snps_total_298,prs_305,risk_ratio_305,num_snps_with_genotypes_305,num_snps_total_305,prs_311,risk_ratio_311,num_snps_with_genotypes_311,num_snps_total_311
0,1000000,1.358896,2.675651,294.0,298.0,1.181928,2.295320,301.0,305.0,1.287001,2.480273,306.0,311.0
1,1000004,-0.311085,0.694429,295.0,298.0,-0.337653,0.681705,302.0,305.0,-0.359895,0.670907,308.0,311.0
2,1000033,-0.650725,0.527822,296.0,298.0,-0.684593,0.516677,303.0,305.0,-0.671097,0.524037,307.0,311.0
3,1000039,1.043029,2.073139,294.0,298.0,0.906361,1.841747,301.0,305.0,0.871333,1.783119,306.0,311.0
4,1000042,-0.115890,0.813016,295.0,298.0,-0.254158,0.728729,302.0,305.0,-0.151900,0.791366,308.0,311.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
414825,9999678,0.324153,1.160002,294.0,298.0,0.411185,1.239993,301.0,305.0,0.398857,1.225389,306.0,311.0
414826,9999697,0.457096,1.291495,294.0,298.0,0.467428,1.296982,301.0,305.0,0.535786,1.366112,307.0,311.0
414827,9999715,0.715247,1.590919,294.0,298.0,0.683479,1.541335,301.0,305.0,0.678675,1.530219,305.0,311.0
414828,9999755,-0.344399,0.675992,294.0,298.0,-0.317667,0.692677,301.0,305.0,-0.264795,0.723522,306.0,311.0


# Exportation des données

In [ ]:
import os
import subprocess

# Enregistre le DataFrame `df_issues` dans un fichier TSV local
destination_filename = 'df_all_prs_298_305_311.csv'
df_all_prs.to_csv(destination_filename, index=False, index_label='index')

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

Trouver un moyen pour importer dans AOU le excel mammorisk (crypter ce fichier, du moins trouver une solution pour le protéger au maximum pour une utilisation sur AOU) 
Dans un second temps -> Refaire ces étapes par éthnie 